# Step 2 — Machine Learning Models

In this step, I train and evaluate multiple machine‑learning models using the cleaned dataset I created in Step 1.  
My goal is to convert the cleaned text into numerical features using TF‑IDF, train at least three ML models, evaluate them, and compare their performance.

This step follows the official capstone requirements:
- Load the cleaned dataset  
- Split into train/test  
- Convert text into numbers using TF‑IDF  
- Train at least 3 ML models  
- Report accuracy, precision, recall, F1-score  
- Plot confusion matrices  
- Create a summary table and choose the best model  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

## 1. Loading the Cleaned Dataset

I load the cleaned dataset generated in Step 1.  
This dataset contains two important columns:

- `clean_text` → the processed text  
- `label` → 0 for real news, 1 for fake news  

In [ ]:
true_df = pd.read_csv("/content/Capstone/Data/true.csv")
fake_df = pd.read_csv("/content/Capstone/Data/fake.csv")

In [ ]:
df = pd.read_csv("cleaned_fake_news.csv")
df.head()

## 2. Train/Test Split

I split the dataset into:
- 80% training data  
- 20% testing data  

I use `stratify=y` to keep the class balance consistent between train and test sets.

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

len(X_train), len(X_test)

## 3. TF‑IDF Vectorization

Since machine‑learning models cannot understand raw text,  
I convert the cleaned text into numerical features using TF‑IDF.

Important rules I follow:
- I use `fit_transform()` on the training set only  
- I use `transform()` on the test set  
- I never fit TF‑IDF on the test set to avoid data leakage  

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_filled = X_train.fillna('')
X_test_filled = X_test.fillna('')

X_train_tfidf = tfidf.fit_transform(X_train_filled)
X_test_tfidf = tfidf.transform(X_test_filled)

X_train_tfidf.shape, X_test_tfidf.shape

## 4. Evaluation Function

To avoid repeating code, I create a helper function that calculates:
- Accuracy  
- Precision  
- Recall  
- F1-score  

These metrics help me compare the performance of all models.

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }

## 5.1 Logistic Regression

Logistic Regression is a strong baseline model for text classification.  
It performs well with high‑dimensional TF‑IDF vectors.

In [ ]:
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_tfidf, y_train)

lr_results = evaluate_model(lr, X_test_tfidf, y_test)
lr_results

In [ ]:
cm = confusion_matrix(y_test, lr.predict(X_test_tfidf))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Logistic Regression Confusion Matrix")
plt.show()

## 5.2 Random Forest

Random Forest is an ensemble model that uses multiple decision trees.  
It can handle noisy data but may struggle with sparse TF‑IDF vectors.

In [ ]:
rf = RandomForestClassifier(n_estimators=200)
rf.fit(X_train_tfidf, y_train)

rf_results = evaluate_model(rf, X_test_tfidf, y_test)
rf_results

In [ ]:
cm = confusion_matrix(y_test, rf.predict(X_test_tfidf))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title("Random Forest Confusion Matrix")
plt.show()

## 5.3 KNN

K‑Nearest Neighbors compares each test sample to the closest training samples.  
It usually performs poorly on high‑dimensional text data,  
but I include it to meet the requirement of training 3 models.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_tfidf, y_train)

knn_results = evaluate_model(knn, X_test_tfidf, y_test)
knn_results

In [ ]:
cm = confusion_matrix(y_test, knn.predict(X_test_tfidf))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
plt.title("KNN Confusion Matrix")
plt.show()

## 6. Summary Table

I combine all model results into one table to compare them side‑by‑side  
and identify the best model based on F1‑score.

In [ ]:
results = pd.DataFrame({
    "Logistic Regression": lr_results,
    "Random Forest": rf_results,
    "KNN": knn_results
})

results.T

## 7. Best Model Conclusion

Based on the F1‑score (the most important metric for imbalanced datasets),  
Logistic Regression performs the best among the three models.

It handles high‑dimensional TF‑IDF vectors efficiently and provides  
the most balanced performance across all metrics.